# Cesvi Indice Infanzia - Esplorazione dati

In [1]:
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("cesvi-indiceinfanzia_data.csv")
df_meta = pd.read_csv("cesvi-indiceinfanzia_metadata.csv")

## Struttura del DataFrame

Il CSV ha 5 colonne:

| Colonna | Valori possibili |
|---|---|
| `territory` | 20 regioni italiane |
| `year` | 2018, 2019, 2020, 2021, 2022, 2024, 2026 |
| `index` | `rischio_adulti`, `rischio_bambini`, `servizi_adulti`, `servizi_bambini` → aggregati: `rischio`, `servizi`, `totale` |
| `capacity` | `cura`, `vita_sana`, `vita_sicura`, `conoscenza_sapere`, `lavorare`, `accedere_risorse` → NaN per gli aggregati |
| `score` | valore numerico (normalizzato) |

### Regola di filtro rapida
- **Solo indici analitici** (no aggregati): `df[df['capacity'].notna()]`
- **Solo aggregati**: `df[df['capacity'].isna()]`


In [31]:
df

,territory,year,capacity,index,score
0,Abruzzo,2018,NaN,rischio,-0.174
1,Basilicata,2018,NaN,rischio,-0.208
2,Calabria,2018,NaN,rischio,-0.683
3,Campania,2018,NaN,rischio,-1.271
4,Emilia-Romagna,2018,NaN,rischio,0.427
...,...,...,...,...,...
2515,Toscana,2026,vita_sicura,servizi_adulti,-0.107
2516,Trentino-Alto Adige/Südtirol,2026,vita_sicura,servizi_adulti,-0.455
2517,Umbria,2026,vita_sicura,servizi_adulti,1.536
2518,Valle d'Aosta/Vallée d'Aoste,2026,vita_sicura,servizi_adulti,0.328


---
## 1. Per anno — confronto regioni in un anno specifico


In [30]:
# Indice TOTALE di tutte le regioni in un anno — bar chart ordinato
anno = 2019
indicatore = "cura"

dff = df[(df["year"] == anno) & (df["capacity"] == indicatore)].sort_values("score", ascending=True)

fig = px.bar(dff, x="score", y="territory", orientation="h",
             title=f"Indice totale per regione — {anno}",
             color="score", color_continuous_scale="RdYlGn",
             labels={"score": "Score", "territory": "Regione"})
fig.show()


In [16]:
# Tutti gli indici (rischio/servizi/totale) per un anno — grouped bar
anno = 2026

dff = df[(df["year"] == anno) & (df["capacity"].isna())]

fig = px.bar(dff, x="territory", y="score", color="index", barmode="group",
             title=f"Indici aggregati per regione — {anno}",
             labels={"score": "Score", "territory": "Regione", "index": "Indice"})
fig.update_layout(xaxis_tickangle=-45)
fig.show()


---
## 2. Per regione — evoluzione nel tempo


In [17]:
# Andamento degli indici aggregati nel tempo per una regione — line chart
regione = "Lombardia"

dff = df[(df["territory"] == regione) & (df["capacity"].isna())]

fig = px.line(dff, x="year", y="score", color="index", markers=True,
              title=f"Andamento indici — {regione}",
              labels={"score": "Score", "year": "Anno", "index": "Indice"})
fig.update_xaxes(tickvals=dff["year"].unique())
fig.show()


In [18]:
# Dettaglio per capacity (indici analitici) per una regione nel tempo
regione = "Lombardia"
indice = "servizi_adulti"  # es. rischio_adulti, servizi_bambini, rischio_bambini

dff = df[(df["territory"] == regione) & (df["index"] == indice)]

fig = px.line(dff, x="year", y="score", color="capacity", markers=True,
              title=f"{indice} per capacità — {regione}",
              labels={"score": "Score", "year": "Anno", "capacity": "Capacità"})
fig.update_xaxes(tickvals=dff["year"].unique())
fig.show()


---
## 3. Per indice/capacità — heatmap regioni × anni


In [19]:
# Heatmap: score TOTALE — righe=regioni, colonne=anni
indice = "totale"  # oppure "rischio", "servizi", "rischio_adulti", ecc.

pivot = (df[df["index"] == indice]
           .pivot_table(index="territory", columns="year", values="score"))

fig = px.imshow(pivot, color_continuous_scale="RdYlGn", aspect="auto",
                title=f"Heatmap score '{indice}' — regioni × anni",
                labels={"color": "Score"})
fig.show()


In [20]:
# Heatmap: score per una capacity specifica — righe=regioni, colonne=anni
capacity = "vita_sana"
indice = "servizi_adulti"

pivot = (df[(df["capacity"] == capacity) & (df["index"] == indice)]
           .pivot_table(index="territory", columns="year", values="score"))

fig = px.imshow(pivot, color_continuous_scale="RdYlGn", aspect="auto",
                title=f"Heatmap '{indice} / {capacity}' — regioni × anni",
                labels={"color": "Score"})
fig.show()


---
## 4. Lookup tabellare — `.loc[]` per filtrare a mano

```python
# Una regione, tutti gli anni e indici
df.set_index(["territory", "year", "index", "capacity"]).loc["Sicilia"]

# Una regione + un anno
df[(df["territory"] == "Sicilia") & (df["year"] == 2026)]

# Un indice + un anno — classifica regioni
df[(df["index"] == "totale") & (df["year"] == 2026)].sort_values("score", ascending=False)

# Un indice + capacity — serie storica per tutte le regioni
df[(df["index"] == "servizi_adulti") & (df["capacity"] == "cura")]
```


In [22]:
df[(df["index"] == "servizi_adulti") & (df["capacity"] == "cura")]


,territory,year,capacity,index,score
160,Abruzzo,2018,cura,servizi_adulti,-0.164
161,Basilicata,2018,cura,servizi_adulti,-0.125
162,Calabria,2018,cura,servizi_adulti,-0.852
163,Campania,2018,cura,servizi_adulti,-1.003
164,Emilia-Romagna,2018,cura,servizi_adulti,1.593
...,...,...,...,...,...
2335,Toscana,2026,cura,servizi_adulti,-0.066
2336,Trentino-Alto Adige/Südtirol,2026,cura,servizi_adulti,0.091
2337,Umbria,2026,cura,servizi_adulti,0.537
2338,Valle d'Aosta/Vallée d'Aoste,2026,cura,servizi_adulti,0.389


## Prove

In [28]:
df

,territory,year,capacity,index,score
0,Abruzzo,2018,NaN,rischio,-0.174
1,Basilicata,2018,NaN,rischio,-0.208
2,Calabria,2018,NaN,rischio,-0.683
3,Campania,2018,NaN,rischio,-1.271
4,Emilia-Romagna,2018,NaN,rischio,0.427
...,...,...,...,...,...
2515,Toscana,2026,vita_sicura,servizi_adulti,-0.107
2516,Trentino-Alto Adige/Südtirol,2026,vita_sicura,servizi_adulti,-0.455
2517,Umbria,2026,vita_sicura,servizi_adulti,1.536
2518,Valle d'Aosta/Vallée d'Aoste,2026,vita_sicura,servizi_adulti,0.328


In [4]:
df.set_index(["territory", "year", "index"]).loc['Veneto']

capacity  score
year index                                   
2018 rischio                       NaN  0.565
     servizi                       NaN  0.438
     totale                        NaN  0.546
     rischio_adulti   accedere_risorse  0.835
     servizi_adulti   accedere_risorse  1.511
...                                ...    ...
2026 rischio_bambini         vita_sana  0.934
     servizi_adulti          vita_sana  0.629
     servizi_bambini         vita_sana -0.504
     rischio_adulti        vita_sicura  0.101
     servizi_adulti        vita_sicura  0.231

[126 rows x 2 columns]

In [10]:
mean = (0.689 + 0.451 + 0.426 + 0.905 + 0.125 + 0.210 + 0.293) / 7
print(mean)

0.4427142857142857
